# 🌟 Anime Video Upscaler (Kaggle)
Este notebook utiliza **Real-ESRGAN (AnimeVideoV3)** para realizar o upscaling de vídeos de anime com alta eficiência e qualidade.
O fluxo utiliza o mesmo padrão de autenticação do Google Drive para importar o vídeo (ex: `saida.mp4`) e exportar o resultado final.


In [2]:
# @title 🚀 1. Setup
import os, subprocess, urllib.request
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

# Instalar Vulkan NA ORDEM CERTA
print("📦 Instalando dependências...")
os.system("apt-get update -qq")
os.system("apt-get remove -y mesa-vulkan-drivers -qq")           # remove Mesa ANTES
os.system("apt-get install -y libvulkan1 vulkan-tools -qq")      # instala base

# Descobrir versão do driver e instalar ICD correto
driver_ver = subprocess.check_output(
    "nvidia-smi --query-gpu=driver_version --format=csv,noheader | head -1",
    shell=True
).decode().strip().split(".")[0]
print(f"🖥️  Driver NVIDIA: {driver_ver}")
os.system(f"apt-get install -y libnvidia-gl-{driver_ver} -qq 2>/dev/null || "
          f"apt-get install -y libnvidia-gl-550 -qq")

# Confirmar ICD
icd = "/usr/share/vulkan/icd.d/nvidia_icd.json"
if os.path.exists(icd):
    os.environ['VK_ICD_FILENAMES'] = icd
    print(f"✅ Vulkan ICD: {icd}")
else:
    print("⚠️  ICD não encontrado — Vulkan pode falhar!")

# Baixar Real-ESRGAN
print("📦 Baixando Real-ESRGAN...")
os.system("wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesrgan-ncnn-vulkan-20220424-ubuntu.zip")
os.system("unzip -o realesrgan-ncnn-vulkan-20220424-ubuntu.zip -d realesrgan > /dev/null")
os.system("chmod +x realesrgan/realesrgan-ncnn-vulkan")

# Google Drive
DRIVE_ACCESS_TOKEN  = "ya29.a0AQvPyIOoPQz8JVUWKwtS_kzmdVexS9g6tC7HUKjSUfrDlp5MoV0_UEZAXunSlTWMJtiNSizuoKAN7JvUaAPVsv0nfXEsZnb-Axv5A6p6IESprQOP_KhWfj08paxVAzp-unaT8thDdPd2qV_ZYzRYho8Q47I9SvS_WCqo641z72wdV1pgbsnQkTo1gZp3KIxZTIeu8K8aCgYKAecSARYSFQHGX2MiQ0sWrZ2QofKcf6vcbeh_IQ0206"
DRIVE_REFRESH_TOKEN = "1//04Svu-32SCa5lCgYIARAAGAQSNwF-L9Ir6qaytT_ktfCAfzDCzLvn-YtfNomsULUaIRwy0nmccGgXizoOcK6Fe6OrYs7zfYi_Q88"

print("📂 Autenticando Google Drive...")
try:
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id="593138121868-16v59bb0bq63sslfnblb4ahs0fbced3f.apps.googleusercontent.com",
        client_secret="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",
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("✅ Google Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"❌ Falha: {e}")

def _buscar_id(caminho):
    partes = caminho.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        q = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        res = drive_service.files().list(q=q, fields="files(id,mimeType)").execute()
        arqs = res.get("files", [])
        if not arqs: return None
        parent_id = arqs[0]["id"]
    return parent_id

def baixar_do_drive(caminho_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        fid = _buscar_id(caminho_drive)
        if not fid: return False
        req = drive_service.files().get_media(fileId=fid)
        with open(destino_local, "wb") as f:
            dl = MediaIoBaseDownload(f, req)
            done = False
            while not done: _, done = dl.next_chunk()
        return True
    except Exception as e:
        print(f"❌ {e}")
        return False

def salvar_no_drive(caminho_local, nome_destino):
    if not drive_service or not os.path.exists(caminho_local): return
    try:
        q = f"name='{nome_destino}' and trashed=false"
        ex = drive_service.files().list(q=q, fields="files(id)").execute().get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if ex:
            drive_service.files().update(fileId=ex[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(body={"name": nome_destino}, media_body=media, fields="id").execute()
        print(f"✅ Salvo no Drive: {nome_destino}")
    except Exception as e:
        print(f"❌ {e}")

# Pastas — tudo absoluto, sem chdir
BASE = "/kaggle/working/upscale"
for d in [f"{BASE}/input", f"{BASE}/frames", f"{BASE}/upscaled",
          f"{BASE}/frames_gpu0", f"{BASE}/frames_gpu1",
          f"{BASE}/upscaled_gpu0", f"{BASE}/upscaled_gpu1"]:
    os.makedirs(d, exist_ok=True)

print("✅ Setup Concluído!")

📦 Instalando dependências...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.


Selecting previously unselected package libvulkan1:amd64.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../libvulkan1_1.3.204.1-2_amd64.deb ...
Unpacking libvulkan1:amd64 (1.3.204.1-2) ...
Selecting previously unselected package mesa-vulkan-drivers:amd64.
Preparing to unpack .../mesa-vulkan-drivers_23.2.1-1ubuntu3.1~22.04.3_amd64.deb ...
Unpacking mesa-vulkan-drivers:amd64 (23.2.1-1ubuntu3.1~22.04.3) ...
Selecting previously unselected package vulkan-tools.
Preparing to unpack .../vulkan-tools_1.3.204.0+dfsg1-1_amd64.deb ...
Unpacking vulkan-tools (1.3.204.0+dfsg1-1) ...
Setting up libvulkan1:amd64 (1.3.204.1-2) ...
Setting up mesa-vulkan-drivers:amd64 (23.2.1-1ubuntu3.1~22.04.3) ...
Setting up vulkan-tools (1.3.204.0+dfsg1-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_5.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a s

In [2]:
# @title 🎬 2. Download e Extração de Frames
import os, subprocess

VIDEO_DRIVE = "KAGGLE/saida.mp4"  # @param {type:"string"}
VIDEO_LOCAL = "/kaggle/working/upscale/input/input.mp4"
FRAMES_DIR  = "/kaggle/working/upscale/frames"

print(f"⬇️  Baixando {VIDEO_DRIVE}...")
ok = baixar_do_drive(VIDEO_DRIVE, VIDEO_LOCAL)
if not ok:
    raise FileNotFoundError(f"Vídeo não encontrado: {VIDEO_DRIVE}")
print("✅ Vídeo baixado!")

os.system(f"rm -f {FRAMES_DIR}/*.png")
print("🎞️  Extraindo frames...")
subprocess.run(
    f"ffmpeg -i {VIDEO_LOCAL} -r 30 -qscale:v 1 -qmin 1 -qmax 1 "
    f"{FRAMES_DIR}/frame_%08d.png -hide_banner -loglevel error",
    shell=True
)
total = len(os.listdir(FRAMES_DIR))
print(f"✅ {total} frames extraídos em {FRAMES_DIR}")

⬇️  Baixando KAGGLE/saida.mp4...
✅ Vídeo baixado!
🎞️  Extraindo frames...
✅ 904 frames extraídos em /kaggle/working/upscale/frames


In [3]:
# @title ⚡ 3. Upscaling Dual T4
import os, subprocess, glob, shutil, threading, time, sys

BASE         = "/kaggle/working/upscale"
FRAMES_DIR   = f"{BASE}/frames"
UPSCALED_DIR = f"{BASE}/upscaled"
REALESRGAN   = f"/kaggle/working/realesrgan/realesrgan-ncnn-vulkan"
MODEL_NAME   = "realesr-animevideov3"
SCALE        = 2

all_frames = sorted(glob.glob(f"{FRAMES_DIR}/*.png"))
total = len(all_frames)
if total == 0:
    raise FileNotFoundError(f"Nenhum frame em {FRAMES_DIR}!")
mid = total // 2

# Limpar e preparar
for d in [f"{BASE}/frames_gpu0", f"{BASE}/frames_gpu1",
          f"{BASE}/upscaled_gpu0", f"{BASE}/upscaled_gpu1", UPSCALED_DIR]:
    os.makedirs(d, exist_ok=True)
    for f in glob.glob(f"{d}/*.png"): os.remove(f)

for f in all_frames[:mid]:
    shutil.copy(f, f"{BASE}/frames_gpu0/")
for f in all_frames[mid:]:
    shutil.copy(f, f"{BASE}/frames_gpu1/")

print(f"📂 Total: {total} | GPU0: {mid} | GPU1: {total-mid}\n")

def run(cmd, log_path):
    with open(log_path, "w") as log:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=log)

t0 = threading.Thread(target=run, args=(
    f"{REALESRGAN} -i {BASE}/frames_gpu0/ -o {BASE}/upscaled_gpu0/ -n {MODEL_NAME} -s {SCALE} -f png -g 0",
    f"{BASE}/log_gpu0.txt"
))
t1 = threading.Thread(target=run, args=(
    f"{REALESRGAN} -i {BASE}/frames_gpu1/ -o {BASE}/upscaled_gpu1/ -n {MODEL_NAME} -s {SCALE} -f png -g 1",
    f"{BASE}/log_gpu1.txt"
))

start = time.time()
t0.start()
t1.start()

def monitor():
    while t0.is_alive() or t1.is_alive():
        d0 = len(glob.glob(f"{BASE}/upscaled_gpu0/*.png"))
        d1 = len(glob.glob(f"{BASE}/upscaled_gpu1/*.png"))
        done = d0 + d1
        elapsed  = time.time() - start
        fps_rate = done / elapsed if elapsed > 0 else 0
        eta      = int((total - done) / fps_rate) if fps_rate > 0 else 0
        pct      = done / total * 100
        bar      = "█" * int(30 * done / total) + "░" * (30 - int(30 * done / total))
        sys.stdout.write(
            f"\r🖥️  GPU0: {d0}/{mid}  |  GPU1: {d1}/{total-mid}  |  "
            f"[{bar}] {done}/{total} ({pct:.1f}%)  |  "
            f"{fps_rate:.2f} f/s  |  ETA: {eta}s   "
        )
        sys.stdout.flush()
        time.sleep(1)

mon = threading.Thread(target=monitor)
mon.start()
t0.join()
t1.join()
mon.join()
print("\n")

out0 = sorted(glob.glob(f"{BASE}/upscaled_gpu0/*.png"))
out1 = sorted(glob.glob(f"{BASE}/upscaled_gpu1/*.png"))
total_gerado = len(out0) + len(out1)

for log_file in [f"{BASE}/log_gpu0.txt", f"{BASE}/log_gpu1.txt"]:
    content = open(log_file).read().strip()
    if content:
        print(f"📋 {os.path.basename(log_file)}:\n{content[-300:]}\n")

if total_gerado < total:
    print(f"❌ {total - total_gerado} frames faltando!")
else:
    for f in out0 + out1:
        shutil.copy(f, UPSCALED_DIR)
    final = len(glob.glob(f"{UPSCALED_DIR}/*.png"))
    elapsed = time.time() - start
    print(f"✅ {final}/{total} frames em {int(elapsed//60)}m{int(elapsed%60)}s")

📂 Total: 904 | GPU0: 452 | GPU1: 452

🖥️  GPU0: 452/452  |  GPU1: 452/452  |  [██████████████████████████████] 904/904 (100.0%)  |  1.41 f/s  |  ETA: 0s    

📋 log_gpu0.txt:
28.33%
30.00%
31.67%
33.33%
35.00%
36.67%
38.33%
40.00%
41.67%
43.33%
45.00%
46.67%
48.33%
50.00%
51.67%
53.33%
55.00%
56.67%
58.33%
60.00%
61.67%
63.33%
65.00%
66.67%
68.33%
70.00%
71.67%
73.33%
75.00%
76.67%
78.33%
80.00%
81.67%
83.33%
85.00%
86.67%
88.33%
90.00%
91.67%
93.33%
95.00%
96.67%
98.33%

📋 log_gpu1.txt:
28.33%
30.00%
31.67%
33.33%
35.00%
36.67%
38.33%
40.00%
41.67%
43.33%
45.00%
46.67%
48.33%
50.00%
51.67%
53.33%
55.00%
56.67%
58.33%
60.00%
61.67%
63.33%
65.00%
66.67%
68.33%
70.00%
71.67%
73.33%
75.00%
76.67%
78.33%
80.00%
81.67%
83.33%
85.00%
86.67%
88.33%
90.00%
91.67%
93.33%
95.00%
96.67%
98.33%



OSError: [Errno 28] No space left on device: '/kaggle/working/upscale/upscaled_gpu1/frame_00000737.png' -> '/kaggle/working/upscale/upscaled/frame_00000737.png'

In [1]:
# @title 🎬 4. Montar Vídeo Final
import os, subprocess, glob, sys, re

BASE         = "/kaggle/working/upscale"
UPSCALED_DIR = f"{BASE}/upscaled"
VIDEO_LOCAL  = f"{BASE}/input/input.mp4"
OUTPUT_FINAL = f"{BASE}/saida_upscaled.mp4"

frames = sorted(glob.glob(f"{UPSCALED_DIR}/*.png"))
total  = len(frames)
print(f"🔍 {total} frames")
if not frames:
    raise FileNotFoundError("Nenhum frame! Rode o upscaling primeiro.")

fps_raw = subprocess.check_output(
    f"ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate "
    f"-of default=noprint_wrappers=1:nokey=1 {VIDEO_LOCAL}", shell=True
).decode().strip()
num, den = (fps_raw.split("/") + ["1"])[:2]
fps_float = float(num) / float(den)
dur_frame = 1.0 / fps_float
print(f"FPS: {fps_float:.4f}")

concat_file = f"{BASE}/concat_list.txt"
with open(concat_file, "w") as f:
    for frame in frames:
        f.write(f"file '{os.path.abspath(frame)}'\n")
        f.write(f"duration {dur_frame:.6f}\n")

def run_ffmpeg(cmd, label):
    print(f"🚀 {label}...")
    proc = subprocess.Popen(cmd, stderr=subprocess.PIPE, universal_newlines=True)
    for line in proc.stderr:
        line = line.strip()
        if "frame=" in line and "fps=" in line:
            fn    = re.search(r"frame=\s*(\d+)", line)
            fps   = re.search(r"fps=\s*([\d.]+)", line)
            speed = re.search(r"speed=\s*([\d.]+x)", line)
            if fn:
                n   = int(fn.group(1))
                pct = n / total * 100
                bar = "█" * int(30 * n / total) + "░" * (30 - int(30 * n / total))
                sys.stdout.write(
                    f"\r🎬 [{bar}] {n}/{total} ({pct:.1f}%)"
                    f"  |  {fps.group(1) if fps else '?'} fps"
                    f"  |  {speed.group(1) if speed else '?'}   "
                )
                sys.stdout.flush()
        elif any(w in line for w in ["Error","error","Invalid"]):
            print(f"\n⚠️  {line}")
    proc.wait()
    print()
    return proc.returncode

ret = run_ffmpeg([
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0", "-i", concat_file,
    "-i", VIDEO_LOCAL,
    "-map", "0:v:0", "-map", "1:a:0?",
    "-c:a", "copy", "-pix_fmt", "yuv420p",
    "-c:v", "h264_nvenc", "-preset", "p4", "-cq", "18",
    "-stats", "-loglevel", "error", OUTPUT_FINAL
], "NVENC (GPU)")

if ret != 0:
    print("⚠️ NVENC falhou, usando libx264...")
    ret = run_ffmpeg([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_file,
        "-i", VIDEO_LOCAL,
        "-map", "0:v:0", "-map", "1:a:0?",
        "-c:a", "copy", "-pix_fmt", "yuv420p",
        "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-threads", "0",
        "-stats", "-loglevel", "error", OUTPUT_FINAL
    ], "libx264 (CPU)")

if ret == 0:
    dur = float(subprocess.check_output(
        f"ffprobe -v error -show_entries format=duration "
        f"-of default=noprint_wrappers=1:nokey=1 {OUTPUT_FINAL}", shell=True
    ).decode().strip())
    esperado = total / fps_float
    print(f"📊 Esperado: {esperado:.1f}s | Resultado: {dur:.1f}s")
    print("✅ Sincronizado!" if abs(dur - esperado) < 1 else "⚠️ Duração divergente!")

🔍 0 frames


FileNotFoundError: Nenhum frame! Rode o upscaling primeiro.

In [ ]:
# @title 📤 6. Salvar no Google Drive
DESTINO_DRIVE = "KAGGLE/saida_upscaled.mp4" # @param {type:"string"}
print(f"Fazendo upload para {DESTINO_DRIVE}...")
salvar_no_drive(OUTPUT_FINAL, DESTINO_DRIVE)
print("🎉 Processo totalmente finalizado!")
